# List of Lagrangians


## Setup


### Imports


In [1]:

import re
import sys
from pathlib import Path
from fractions import Fraction

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from symbolica import Expression, S

from model import (
    COLOR_ADJ_INDEX,
    COLOR_FUND_INDEX,
    LORENTZ_INDEX,
    SPINOR_INDEX,
    WEAK_ADJ_INDEX,
    WEAK_FUND_INDEX,
    ComplexScalarKineticTerm,
    CovD,
    DiracKineticTerm,
    Field,
    FieldStrength,
    Gamma,
    Gamma5,
    GaugeFixing,
    GaugeGroup,
    GaugeKineticTerm,
    GaugeRepresentation,
    GhostField,
    GhostLagrangian,
    Model,
    PartialD,
)
from symbolic.spenso_structures import (
    gauge_generator,
    structure_constant,
    weak_gauge_generator,
    weak_structure_constant,
)
from symbolic.vertex_engine import I, compact_sum_notation, compact_vertex_sum_form

ANSI_ESCAPE_RE = re.compile(r"\x1B\[[0-?]*[ -/]*[@-~]")


def expr_text(expr):
    text = expr.to_canonical_string() if hasattr(expr, "to_canonical_string") else str(expr)
    return text.replace("spenso::", "")


def clean(text):
    return ANSI_ESCAPE_RE.sub("", str(text))


def signature_rows(signatures):
    return tuple(
        {
            "fields": ", ".join(signature.names),
            "arity": signature.arity,
            "terms": signature.term_count,
            "sectors": ", ".join(signature.sectors),
        }
        for signature in signatures
    )


def report_row(report):
    return {
        "matched_signatures": report.matched_signatures,
        "matched_terms": report.matched_terms,
        "total_signatures": report.total_signatures,
        "total_terms": report.total_terms,
        "signatures": signature_rows(report.signatures),
    }


def validation_rows(report):
    return {
        "ok": report.ok,
        "issues": tuple(
            {
                "code": issue.code,
                "severity": issue.severity,
                "message": issue.message,
            }
            for issue in report.issues
        ),
    }


def show(title, result):
    print("==========")
    print(title)
    if isinstance(result, dict):
        print(f"{len(result)} vertex signature(s)")
        print()
        for signature, expression in result.items():
            print("Vertex:", signature)
            print("Rule:", clean(expression))
            print()
    else:
        print(clean(result))
        print()


def show_model(model, *fields, compact_form=None, sum_notation=None):
    source_terms = model.source_lagrangian_terms()
    if source_terms:
        show("Lagrangian", source_terms[0])
    lagrangian = model.lagrangian()
    if fields:
        show("Feynman Rule", lagrangian.feynman_rule(*fields, include_delta=False))
    else:
        show("Feynman Rules", lagrangian.feynman_rules(include_delta=False))
    if compact_form is not None:
        show("Compact Form", compact_form)
    if sum_notation is not None:
        show("Sum Notation", sum_notation)


### Symbols


In [2]:
mu, nu, rho = S("mu"), S("nu"), S("rho")
eQED, gS, g2, xiQCD = S("eQED"), S("gS"), S("g2"), S("xiQCD")
qPhi, qPsi, qQ, YL = S("qPhi"), S("qPsi"), S("qQ"), S("YL")
lam, y, g4, gGamma = S("lam"), S("y"), S("g4"), S("gGamma")
lam4, g, lamC, gD2, gijk = S("lam4"), S("g"), S("lamC"), S("gD2"), S("gijk")
yF, gV, gJJ, g_psi4, g1, g2f = S("yF"), S("gV"), S("gJJ"), S("g_psi4"), S("g1"), S("g2f")
i, j, k = S("i"), S("j"), S("k")
YH = S("YH")
xiW = S("xiW")


### Fields


In [3]:

Phi = Field(
    "Phi",
    spin=0,
    self_conjugate=False,
    symbol=S("phi"),
    conjugate_symbol=S("phidag"),
    quantum_numbers={"Q": qPhi},
)
Chi = Field(
    "Chi",
    spin=0,
    self_conjugate=False,
    symbol=S("chi"),
    conjugate_symbol=S("chidag"),
)
Psi = Field(
    "Psi",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=S("psi"),
    conjugate_symbol=S("psibar"),
    indices=(SPINOR_INDEX,),
    quantum_numbers={"Q": qPsi},
)
Xi = Field(
    "Xi",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=S("xi"),
    conjugate_symbol=S("xibar"),
    indices=(SPINOR_INDEX,),
)
Quark = Field(
    "q",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=S("q"),
    conjugate_symbol=S("qbar"),
    indices=(SPINOR_INDEX, COLOR_FUND_INDEX),
    quantum_numbers={"Q": qQ},
)
Photon = Field("A", spin=1, self_conjugate=True, symbol=S("A0"), indices=(LORENTZ_INDEX,))
Gluon = Field("G", spin=1, self_conjugate=True, symbol=S("G0"), indices=(LORENTZ_INDEX, COLOR_ADJ_INDEX))
GhostG = GhostField(
    "ghG",
    ghost_of=Gluon,
    self_conjugate=False,
    symbol=S("ghG0"),
    conjugate_symbol=S("ghGbar0"),
    indices=(COLOR_ADJ_INDEX,),
    quantum_numbers={"GhostNumber": 1},
)
W = Field("W", spin=1, self_conjugate=True, symbol=S("W0"), indices=(LORENTZ_INDEX, WEAK_ADJ_INDEX))
GhostW = GhostField(
    "ghW",
    ghost_of=W,
    self_conjugate=False,
    symbol=S("ghW0"),
    conjugate_symbol=S("ghWbar0"),
    indices=(WEAK_ADJ_INDEX,),
    quantum_numbers={"GhostNumber": 1},
)
L = Field(
    "L",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=S("L0"),
    conjugate_symbol=S("Lbar0"),
    indices=(SPINOR_INDEX, WEAK_FUND_INDEX),
    quantum_numbers={"Y": YL},
)

B = Field(
    "B",
    spin=1,
    self_conjugate=True,
    symbol=S("B0"),
    indices=(LORENTZ_INDEX,),
)
HY = Field(
    "HY",
    spin=0,
    self_conjugate=False,
    symbol=S("HY0"),
    conjugate_symbol=S("HYdag0"),
    indices=(WEAK_FUND_INDEX,),
    quantum_numbers={"Y": YH},
)


# Compact scalar-only species for the short model-layer examples below.
PhiR = Field("PhiR", spin=0, self_conjugate=True, symbol=S("phi0"))
ChiR = Field("ChiR", spin=0, self_conjugate=True, symbol=S("chi0"))
PhiC = Field("PhiC", spin=0, self_conjugate=False, symbol=S("phiC0"), conjugate_symbol=S("phiCdag0"))
Phi_i = Field("phi_i", spin=0, self_conjugate=True, symbol=S("phi_i0"))
Phi_j = Field("phi_j", spin=0, self_conjugate=True, symbol=S("phi_j0"))
Phi_k = Field("phi_k", spin=0, self_conjugate=True, symbol=S("phi_k0"))

(Phi, Chi, Psi, Xi, Quark, Photon, Gluon, GhostG, W, GhostW, HY, L)


(Field(name='Phi', spin=0, self_conjugate=False, indices=(), kind='scalar', statistics='boson', symbol=phi, conjugate_symbol=phidag, mass=None, quantum_numbers={'Q': qPhi}, ghost_of=None),
 Field(name='Chi', spin=0, self_conjugate=False, indices=(), kind='scalar', statistics='boson', symbol=chi, conjugate_symbol=chidag, mass=None, quantum_numbers={}, ghost_of=None),
 Field(name='Psi', spin=Fraction(1, 2), self_conjugate=False, indices=(IndexType(name='Spinor', representation=Representation { rep: SelfDual(1), dim: Concrete(4) }, kind='spinor', prefix='i'),), kind='fermion', statistics='fermion', symbol=psi, conjugate_symbol=psibar, mass=None, quantum_numbers={'Q': qPsi}, ghost_of=None),
 Field(name='Xi', spin=Fraction(1, 2), self_conjugate=False, indices=(IndexType(name='Spinor', representation=Representation { rep: SelfDual(1), dim: Concrete(4) }, kind='spinor', prefix='i'),), kind='fermion', statistics='fermion', symbol=xi, conjugate_symbol=xibar, mass=None, quantum_numbers={}, ghost

### Gauge Representations


In [4]:
COLOR_FUND_REP = GaugeRepresentation(
    index=COLOR_FUND_INDEX,
    generator_builder=gauge_generator,
    name="fund",
)
WEAK_DOUBLET_REP = GaugeRepresentation(
    index=WEAK_FUND_INDEX,
    generator_builder=weak_gauge_generator,
    name="doublet",
)

(COLOR_FUND_REP, WEAK_DOUBLET_REP)


(GaugeRepresentation(index=IndexType(name='ColorFund', representation=Representation { rep: Dualizable(3), dim: Concrete(3) }, kind='color_fund', prefix='c'), generator_builder=<function gauge_generator at 0x10c6087d0>, name='fund', slot=None, slot_policy='unique'),
 GaugeRepresentation(index=IndexType(name='WeakFund', representation=Representation { rep: Dualizable(3), dim: Concrete(2) }, kind='weak_fund', prefix='w'), generator_builder=<function weak_gauge_generator at 0x10c608930>, name='doublet', slot=None, slot_policy='unique'))

### Gauge Groups


In [5]:
U1QED = GaugeGroup(
    name="U1QED",
    abelian=True,
    coupling=eQED,
    gauge_boson=Photon.symbol,
    charge="Q",
)
U1Y = GaugeGroup(
    name="U1Y",
    abelian=True,
    coupling=g1,
    gauge_boson=B.symbol,
    charge="Y",
)
SU3C = GaugeGroup(
    name="SU3C",
    abelian=False,
    coupling=gS,
    gauge_boson=Gluon.symbol,
    ghost_field=GhostG.symbol,
    structure_constant=structure_constant,
    representations=(COLOR_FUND_REP,),
)
SU2L = GaugeGroup(
    name="SU2L",
    abelian=False,
    coupling=g2,
    gauge_boson=W.symbol,
    ghost_field=GhostW.symbol,
    structure_constant=weak_structure_constant,
    representations=(WEAK_DOUBLET_REP,),
)

(U1QED, SU3C, SU2L)


(GaugeGroup(name='U1QED', abelian=True, coupling=eQED, gauge_boson=A0, ghost_field=None, structure_constant=None, representations=(), charge='Q'),
 GaugeGroup(name='SU3C', abelian=False, coupling=gS, gauge_boson=G0, ghost_field=ghG0, structure_constant=<function structure_constant at 0x10c608880>, representations=(GaugeRepresentation(index=IndexType(name='ColorFund', representation=Representation { rep: Dualizable(3), dim: Concrete(3) }, kind='color_fund', prefix='c'), generator_builder=<function gauge_generator at 0x10c6087d0>, name='fund', slot=None, slot_policy='unique'),), charge=None),
 GaugeGroup(name='SU2L', abelian=False, coupling=g2, gauge_boson=W0, ghost_field=ghW0, structure_constant=<function weak_structure_constant at 0x10c6089e0>, representations=(GaugeRepresentation(index=IndexType(name='WeakFund', representation=Representation { rep: Dualizable(3), dim: Concrete(2) }, kind='weak_fund', prefix='w'), generator_builder=<function weak_gauge_generator at 0x10c608930>, name='

## Scalar Examples


In [6]:
phi4 = Model(fields=(PhiR,), lagrangian_decl=lam4 * PhiR * PhiR * PhiR * PhiR)
show_model(phi4)

phi2chi2 = Model(fields=(PhiR, ChiR), lagrangian_decl=g * PhiR * PhiR * ChiR * ChiR)
show_model(phi2chi2)

complex_bilinear = Model(fields=(PhiC,), lagrangian_decl=lamC * PhiC.bar * PhiC)
show_model(complex_bilinear)

multi_species = Model(
    fields=(Phi_i, Phi_j, Phi_k),
    lagrangian_decl=gijk(i, j, k) * Phi_i * Phi_i * Phi_j * Phi_j * Phi_k * Phi_k,
)
show_model(multi_species)


Lagrangian
lam4 * PhiR * PhiR * PhiR * PhiR

Feynman Rules
1 vertex signature(s)

Vertex: ('PhiR', 'PhiR', 'PhiR', 'PhiR')
Rule: 24𝑖*lam4

Lagrangian
g * PhiR * PhiR * ChiR * ChiR

Feynman Rules
1 vertex signature(s)

Vertex: ('PhiR', 'PhiR', 'ChiR', 'ChiR')
Rule: 4𝑖*g

Lagrangian
lamC * PhiC.bar * PhiC

Feynman Rules
1 vertex signature(s)

Vertex: ('PhiC.bar', 'PhiC')
Rule: 1𝑖*lamC

Lagrangian
gijk(i,j,k) * phi_i * phi_i * phi_j * phi_j * phi_k * phi_k

Feynman Rules
1 vertex signature(s)

Vertex: ('phi_i', 'phi_i', 'phi_j', 'phi_j', 'phi_k', 'phi_k')
Rule: 8𝑖*gijk(i,j,k)



In [7]:

derivative_phi4 = Model(
    fields=(PhiR,),
    lagrangian_decl=gD2 * PartialD(PhiR, mu) * PartialD(PhiR, mu) * PhiR * PhiR,
)
show_model(derivative_phi4)

Lagrangian
gD2 * PartialD(PhiR, mu) * PartialD(PhiR, mu) * PhiR * PhiR

Feynman Rules
1 vertex signature(s)

Vertex: ('PhiR', 'PhiR', 'PhiR', 'PhiR')
Rule: -4𝑖*gD2*pcomp(q1,mu1_int)*pcomp(q2,mu1_int)-4𝑖*gD2*pcomp(q1,mu1_int)*pcomp(q3,mu1_int)-4𝑖*gD2*pcomp(q1,mu1_int)*pcomp(q4,mu1_int)-4𝑖*gD2*pcomp(q2,mu1_int)*pcomp(q3,mu1_int)-4𝑖*gD2*pcomp(q2,mu1_int)*pcomp(q4,mu1_int)-4𝑖*gD2*pcomp(q3,mu1_int)*pcomp(q4,mu1_int)



## Fermion Examples


In [8]:
yukawa_model = Model(fields=(Psi, Phi), lagrangian_decl=yF * Psi.bar * Psi * Phi)
show_model(yukawa_model)

vector_current_model = Model(fields=(Psi, Photon), lagrangian_decl=gV * Psi.bar * Gamma(mu) * Psi * Photon)
show_model(vector_current_model)

axial_current_model = Model(
    fields=(Psi, Photon),
    lagrangian_decl=gV * Psi.bar * Gamma(mu) * Gamma5() * Psi * Photon,
)
show_model(axial_current_model)

four_fermion_model = Model(
    fields=(Psi,),
    lagrangian_decl=-(g_psi4 / Expression.num(2)) * Psi.bar * Psi * Psi.bar * Psi,
)
show_model(four_fermion_model)

current_current_model = Model(
    fields=(Psi,),
    lagrangian_decl=gJJ * Psi.bar * Gamma(mu) * Psi * Psi.bar * Gamma(mu) * Psi,
)
show_model(current_current_model)

dpsibar_model = Model(
    fields=(Psi, Phi),
    lagrangian_decl=yF * PartialD(Psi.bar, mu) * Psi * Phi * Phi,
)
show_model(dpsibar_model)

dpsi_model = Model(
    fields=(Psi, Phi, Chi),
    lagrangian_decl=yF * Psi.bar * PartialD(Psi, nu) * Phi * Chi,
)
show_model(dpsi_model)

dphi_dchi_model = Model(
    fields=(Psi, Phi, Chi),
    lagrangian_decl=yF * Psi.bar * Psi * PartialD(Phi, mu) * PartialD(Chi, nu),
)
show_model(dphi_dchi_model)

d2phi_chi_model = Model(
    fields=(Psi, Phi, Chi),
    lagrangian_decl=g1 * Psi.bar * Psi * PartialD(PartialD(Phi, mu), mu) * Chi,
)
show_model(d2phi_chi_model)

d2phi2_model = Model(
    fields=(Psi, Phi),
    lagrangian_decl=(
        g2f
        * Psi.bar
        * Psi
        * PartialD(PartialD(Phi, mu), nu)
        * PartialD(PartialD(Phi, mu), nu)
    ),
)
show_model(d2phi2_model)


Lagrangian
yF * Psi.bar * Psi * Phi

Feynman Rules
1 vertex signature(s)

Vertex: ('Psi.bar', 'Psi', 'Phi')
Rule: 1𝑖*yF*g(bis(4, i1),bis(4, i2))

Lagrangian
gV * Psi.bar * Gamma(mu) * Psi * A

Feynman Rules
1 vertex signature(s)

Vertex: ('Psi.bar', 'Psi', 'A')
Rule: 1𝑖*gV*gamma(bis(4, i1),bis(4, i2),mink(4, mu3))

Lagrangian
gV * Psi.bar * Gamma(mu) * Gamma5() * Psi * A

Feynman Rules
1 vertex signature(s)

Vertex: ('Psi.bar', 'Psi', 'A')
Rule: 1𝑖*gV*gamma(bis(4, i1),bis(4, spinor_decl_3),mink(4, mu3))*gamma5(bis(4, spinor_decl_3),bis(4, i2))

Lagrangian
-1/2*g_psi4 * Psi.bar * Psi * Psi.bar * Psi

Feynman Rules
1 vertex signature(s)

Vertex: ('Psi.bar', 'Psi', 'Psi.bar', 'Psi')
Rule: -1𝑖*g_psi4*g(bis(4, i1),bis(4, i2))*g(bis(4, i3),bis(4, i4))-1𝑖*g_psi4*g(bis(4, i1),bis(4, i4))*g(bis(4, i2),bis(4, i3))

Lagrangian
gJJ * Psi.bar * Gamma(mu) * Psi * Psi.bar * Gamma(mu) * Psi

Feynman Rules
1 vertex signature(s)

Vertex: ('Psi.bar', 'Psi', 'Psi.bar', 'Psi')
Rule: 2𝑖*gJJ*gamma(bis(4, i1)

## Gauge Examples


In [9]:

qed_fermion_model = Model(
    gauge_groups=(U1QED,),
    fields=(Psi, Photon),
    lagrangian_decl=I * Psi.bar * Gamma(mu) * CovD(Psi, mu),
)
show_model(qed_fermion_model, Psi.bar, Psi, Photon)

qcd_fermion_model = Model(
    gauge_groups=(SU3C,),
    fields=(Quark, Gluon),
    lagrangian_decl=I * Quark.bar * Gamma(mu) * CovD(Quark, mu),
)
show_model(qcd_fermion_model, Quark.bar, Quark, Gluon)

scalar_qed_model = Model(
    gauge_groups=(U1QED,),
    fields=(Phi, Photon),
    lagrangian_decl=CovD(Phi.bar, mu) * CovD(Phi, mu),
)
show_model(scalar_qed_model, Phi.bar, Phi, Photon)
show_model(scalar_qed_model, Phi.bar, Phi, Photon, Photon)

photon_kinetic_model = Model(
    gauge_groups=(U1QED,),
    fields=(Photon,),
    lagrangian_decl=-(Expression.num(1) / Expression.num(4)) * FieldStrength(U1QED, mu, nu) * FieldStrength(U1QED, mu, nu),
)
show_model(photon_kinetic_model, Photon, Photon)

gluon_kinetic_model = Model(
    gauge_groups=(SU3C,),
    fields=(Gluon,),
    lagrangian_decl=-(Expression.num(1) / Expression.num(4)) * FieldStrength(SU3C, mu, nu) * FieldStrength(SU3C, mu, nu),
)
show_model(gluon_kinetic_model, Gluon, Gluon, Gluon)

gauge_fixing_model = Model(
    gauge_groups=(U1QED,),
    fields=(Photon,),
    lagrangian_decl=GaugeFixing(U1QED, xi=S("xi")),
)
show_model(gauge_fixing_model, Photon, Photon)

ghost_model = Model(
    gauge_groups=(SU3C,),
    fields=(Gluon, GhostG),
    lagrangian_decl=GhostLagrangian(SU3C),
)
show_model(ghost_model, GhostG.bar, Gluon, GhostG)

qcd_ghost_model_manual = Model(
    gauge_groups=(SU3C,),
    fields=(Gluon, GhostG),
    lagrangian_decl=-(GhostG.bar * PartialD(CovD(GhostG, mu), mu)),
)
show_model(qcd_ghost_model_manual, GhostG.bar, GhostG)
show_model(qcd_ghost_model_manual, GhostG.bar, Gluon, GhostG)

Lagrangian
1𝑖 * Psi.bar * Gamma(mu) * CovD(Psi, mu)

Feynman Rule
1𝑖*eQED*qPsi*gamma(bis(4, i1),bis(4, i2),mink(4, mu3))

Lagrangian
1𝑖 * q.bar * Gamma(mu) * CovD(q, mu)

Feynman Rule
1𝑖*gS*gamma(bis(4, i1),bis(4, i2),mink(4, mu3))*t(coad(8, a3),cof(3, c1),cof(3, c2))

Lagrangian
CovD(Phi.bar, mu) * CovD(Phi, mu)

Feynman Rule
-1𝑖*eQED*qPhi*pcomp(q1,mu)+1𝑖*eQED*qPhi*pcomp(q2,mu)

Lagrangian
CovD(Phi.bar, mu) * CovD(Phi, mu)

Feynman Rule
2𝑖*eQED^2*qPhi^2*g(mink(4, mu3),mink(4, mu4))

Lagrangian
-1/4 * FieldStrength(U1QED, mu, nu) * FieldStrength(U1QED, mu, nu)

Feynman Rule
-1𝑖*pcomp(q1,mu2)*pcomp(q2,mu1)+1𝑖*g(mink(4, mu1),mink(4, mu2))*pcomp(q1,mu1_int)*pcomp(q2,mu1_int)

Lagrangian
-1/4 * FieldStrength(SU3C, mu, nu) * FieldStrength(SU3C, mu, nu)

Feynman Rule
-gS*g(mink(4, mu3),mink(4, mu1))*f(coad(8, a3),coad(8, a1),coad(8, a2))*pcomp(q1,mu2)+gS*g(mink(4, mu3),mink(4, mu1))*f(coad(8, a3),coad(8, a1),coad(8, a2))*pcomp(q3,mu2)+gS*g(mink(4, mu3),mink(4, mu2))*f(coad(8, a3),coad(8, a1)

In [10]:

xiQED = S("xiQED")

qed_gf_helper = Model(
    gauge_groups=(U1QED,),
    fields=(Photon,),
    lagrangian_decl=GaugeFixing(U1QED, xi=xiQED),
)
qed_gf_manual = Model(
    gauge_groups=(U1QED,),
    fields=(Photon,),
    lagrangian_decl=(
        -(Expression.num(1) / (Expression.num(2) * xiQED))
        * PartialD(Photon(S("mu_qed_gf")), S("mu_qed_gf"))
        * PartialD(Photon(S("nu_qed_gf")), S("nu_qed_gf"))
    ),
)

show(
    "QED Gauge-Fixing Helper Rule",
    qed_gf_helper.lagrangian().feynman_rule(Photon, Photon, include_delta=False),
)
show(
    "QED Gauge-Fixing Manual Rule",
    qed_gf_manual.lagrangian().feynman_rule(Photon, Photon, include_delta=False),
)

qcd_gf_helper = Model(
    gauge_groups=(SU3C,),
    fields=(Gluon,),
    lagrangian_decl=GaugeFixing(SU3C, xi=xiQCD),
)
qcd_gf_manual = Model(
    gauge_groups=(SU3C,),
    fields=(Gluon,),
    lagrangian_decl=(
        -(Expression.num(1) / (Expression.num(2) * xiQCD))
        * PartialD(Gluon(S("mu_qcd_gf"), S("a_qcd_gf")), S("mu_qcd_gf"))
        * PartialD(Gluon(S("nu_qcd_gf"), S("a_qcd_gf")), S("nu_qcd_gf"))
    ),
)

show(
    "QCD Gauge-Fixing Helper Rule",
    qcd_gf_helper.lagrangian().feynman_rule(Gluon, Gluon, include_delta=False),
)
show(
    "QCD Gauge-Fixing Manual Rule",
    qcd_gf_manual.lagrangian().feynman_rule(Gluon, Gluon, include_delta=False),
)

qcd_gf_ghost_direct = Model(
    gauge_groups=(SU3C,),
    fields=(Gluon, GhostG),
    lagrangian_decl=GaugeFixing(SU3C, xi=xiQCD) - GhostG.bar * PartialD(CovD(GhostG, mu), mu),
)
show(
    "QCD Gauge-Fixing + Ghost Rules",
    qcd_gf_ghost_direct.lagrangian().feynman_rules(include_delta=False),
)


QED Gauge-Fixing Helper Rule
1𝑖*pcomp(q1,mu1)*pcomp(q2,mu2)/xiQED

QED Gauge-Fixing Manual Rule
1𝑖*pcomp(q1,mu1)*pcomp(q2,mu2)/xiQED

QCD Gauge-Fixing Helper Rule
1𝑖*g(coad(8, a1),coad(8, a2))*pcomp(q1,mu1)*pcomp(q2,mu2)/xiQCD

QCD Gauge-Fixing Manual Rule
1𝑖*g(coad(8, a1),coad(8, a2))*pcomp(q1,mu1)*pcomp(q2,mu2)/xiQCD

QCD Gauge-Fixing + Ghost Rules
3 vertex signature(s)

Vertex: ('G', 'G')
Rule: 1𝑖*g(coad(8, a1),coad(8, a2))*pcomp(q1,mu1)*pcomp(q2,mu2)/xiQCD

Vertex: ('ghG.bar', 'ghG')
Rule: 1𝑖*g(coad(8, a1),coad(8, a2))*pcomp(q2,mu1_int)^2

Vertex: ('ghG.bar', 'ghG', 'G')
Rule: gS*f(coad(8, a3),coad(8, a1),coad(8, a2))*pcomp(q2,mu3)+gS*f(coad(8, a3),coad(8, a1),coad(8, a2))*pcomp(q3,mu3)



## SU(2) Examples


In [11]:
su2_fermion_model = Model(
    gauge_groups=(SU2L,),
    fields=(L, W),
    lagrangian_decl=I * L.bar * Gamma(mu) * CovD(L, mu),
)
show_model(su2_fermion_model)

su2_scalar_model = Model(
    gauge_groups=(SU2L,),
    fields=(HY, W),
    lagrangian_decl=CovD(HY.bar, mu) * CovD(HY, mu),
)
show_model(su2_scalar_model)

su2_ym_model = Model(
    gauge_groups=(SU2L,),
    fields=(W,),
    lagrangian_decl=-(Expression.num(1) / Expression.num(4)) * FieldStrength(SU2L, mu, nu) * FieldStrength(SU2L, mu, nu),
)
show_model(su2_ym_model)

su2_gf_ghost_model = Model(
    gauge_groups=(SU2L,),
    fields=(W, GhostW),
    lagrangian_decl=GaugeFixing(SU2L, xi=xiW) - GhostW.bar * PartialD(CovD(GhostW, mu), mu),
)
show_model(su2_gf_ghost_model)

ew_fermion_model = Model(
    gauge_groups=(SU2L, U1Y),
    fields=(L, W, B),
    lagrangian_decl=I * L.bar * Gamma(mu) * CovD(L, mu),
)
show_model(ew_fermion_model)

ew_scalar_model = Model(
    gauge_groups=(SU2L, U1Y),
    fields=(HY, W, B),
    lagrangian_decl=CovD(HY.bar, mu) * CovD(HY, mu),
)
show_model(ew_scalar_model)


Lagrangian
1𝑖 * L.bar * Gamma(mu) * CovD(L, mu)

Feynman Rules
2 vertex signature(s)

Vertex: ('L.bar', 'L', 'W')
Rule: 1𝑖*g2*gamma(bis(4, i1),bis(4, i2),mink(4, mu3))*t(coad(3, aw3),cof(2, w1),cof(2, w2))

Vertex: ('L.bar', 'L')
Rule: 1𝑖*g(cof(2, w1),cof(2, w2))*gamma(bis(4, i1),bis(4, i2),mink(4, mu1_int))*pcomp(q2,mu1_int)

Lagrangian
CovD(HY.bar, mu) * CovD(HY, mu)

Feynman Rules
3 vertex signature(s)

Vertex: ('HY.bar', 'HY', 'W')
Rule: -1𝑖*g2*t(coad(3, aw3),cof(2, w1),cof(2, w2))*pcomp(q1,mu)+1𝑖*g2*t(coad(3, aw3),cof(2, w1),cof(2, w2))*pcomp(q2,mu)

Vertex: ('HY.bar', 'HY', 'W', 'W')
Rule: 1𝑖*g2^2*g(mink(4, mu3),mink(4, mu4))*t(coad(3, aw3),cof(2, w1),cof(2, w_mid_HY_SU2L))*t(coad(3, aw4),cof(2, w_mid_HY_SU2L),cof(2, w2))+1𝑖*g2^2*g(mink(4, mu3),mink(4, mu4))*t(coad(3, aw3),cof(2, w_mid_HY_SU2L),cof(2, w2))*t(coad(3, aw4),cof(2, w1),cof(2, w_mid_HY_SU2L))

Vertex: ('HY.bar', 'HY')
Rule: -1𝑖*g(cof(2, w1),cof(2, w2))*pcomp(q1,mu1_int)*pcomp(q2,mu1_int)

Lagrangian
-1/4 * FieldStreng

## Standard Model Gauge Structure


In [12]:
YqL = Expression.num(1) / Expression.num(6)
YuR = Expression.num(2) / Expression.num(3)
YdR = -(Expression.num(1) / Expression.num(3))
YlL = -(Expression.num(1) / Expression.num(2))
YeR = -Expression.num(1)
YHSM = Expression.num(1) / Expression.num(2)

# TODO: the current API does not encode chiral projectors explicitly.
# This section tests the full SM gauge-representation structure through CovD(...).
qL = Field(
    "qL",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=S("qL0"),
    conjugate_symbol=S("qLbar0"),
    indices=(SPINOR_INDEX, COLOR_FUND_INDEX, WEAK_FUND_INDEX),
    quantum_numbers={"Y": YqL},
)
uR = Field(
    "uR",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=S("uR0"),
    conjugate_symbol=S("uRbar0"),
    indices=(SPINOR_INDEX, COLOR_FUND_INDEX),
    quantum_numbers={"Y": YuR},
)
dR = Field(
    "dR",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=S("dR0"),
    conjugate_symbol=S("dRbar0"),
    indices=(SPINOR_INDEX, COLOR_FUND_INDEX),
    quantum_numbers={"Y": YdR},
)
lL = Field(
    "lL",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=S("lL0"),
    conjugate_symbol=S("lLbar0"),
    indices=(SPINOR_INDEX, WEAK_FUND_INDEX),
    quantum_numbers={"Y": YlL},
)
eR = Field(
    "eR",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=S("eR0"),
    conjugate_symbol=S("eRbar0"),
    indices=(SPINOR_INDEX,),
    quantum_numbers={"Y": YeR},
)
H = Field(
    "H",
    spin=0,
    self_conjugate=False,
    symbol=S("HSM0"),
    conjugate_symbol=S("HSMdag0"),
    indices=(WEAK_FUND_INDEX,),
    quantum_numbers={"Y": YHSM},
)

G = Gluon

sm_gauge_model = Model(
    gauge_groups=(SU3C, SU2L, U1Y),
    fields=(qL, uR, dR, lL, eR, H, G, W, B),
    lagrangian_decl=(
        I * qL.bar * Gamma(mu) * CovD(qL, mu)
        + I * uR.bar * Gamma(mu) * CovD(uR, mu)
        + I * dR.bar * Gamma(mu) * CovD(dR, mu)
        + I * lL.bar * Gamma(mu) * CovD(lL, mu)
        + I * eR.bar * Gamma(mu) * CovD(eR, mu)
        + CovD(H.bar, mu) * CovD(H, mu)
    ),
)
show_model(sm_gauge_model)


Lagrangian
1𝑖 * qL.bar * Gamma(mu) * CovD(qL, mu)

Feynman Rules
21 vertex signature(s)

Vertex: ('qL.bar', 'qL', 'G')
Rule: 1𝑖*gS*g(cof(2, w1),cof(2, w2))*gamma(bis(4, i1),bis(4, i2),mink(4, mu3))*t(coad(8, a3),cof(3, c1),cof(3, c2))

Vertex: ('qL.bar', 'qL', 'W')
Rule: 1𝑖*g2*g(cof(3, c1),cof(3, c2))*gamma(bis(4, i1),bis(4, i2),mink(4, mu3))*t(coad(3, aw3),cof(2, w1),cof(2, w2))

Vertex: ('qL.bar', 'qL', 'B')
Rule: 1𝑖/6*g1*g(cof(2, w1),cof(2, w2))*g(cof(3, c1),cof(3, c2))*gamma(bis(4, i1),bis(4, i2),mink(4, mu3))

Vertex: ('qL.bar', 'qL')
Rule: 1𝑖*g(cof(2, w1),cof(2, w2))*g(cof(3, c1),cof(3, c2))*gamma(bis(4, i1),bis(4, i2),mink(4, mu1_int))*pcomp(q2,mu1_int)

Vertex: ('uR.bar', 'uR', 'G')
Rule: 1𝑖*gS*gamma(bis(4, i1),bis(4, i2),mink(4, mu3))*t(coad(8, a3),cof(3, c1),cof(3, c2))

Vertex: ('uR.bar', 'uR', 'B')
Rule: 2𝑖/3*g1*g(cof(3, c1),cof(3, c2))*gamma(bis(4, i1),bis(4, i2),mink(4, mu3))

Vertex: ('uR.bar', 'uR')
Rule: 1𝑖*g(cof(3, c1),cof(3, c2))*gamma(bis(4, i1),bis(4, i2),mink(4, m

In [13]:
sm_full_gauge_model = Model(
    gauge_groups=(SU3C, SU2L, U1Y),
    fields=(qL, uR, dR, lL, eR, H, G, W, B),
    lagrangian_decl=(
        I * qL.bar * Gamma(mu) * CovD(qL, mu)
        + I * uR.bar * Gamma(mu) * CovD(uR, mu)
        + I * dR.bar * Gamma(mu) * CovD(dR, mu)
        + I * lL.bar * Gamma(mu) * CovD(lL, mu)
        + I * eR.bar * Gamma(mu) * CovD(eR, mu)
        + CovD(H.bar, mu) * CovD(H, mu)
        - (Expression.num(1) / Expression.num(4)) * FieldStrength(SU3C, mu, nu) * FieldStrength(SU3C, mu, nu)
        - (Expression.num(1) / Expression.num(4)) * FieldStrength(SU2L, mu, nu) * FieldStrength(SU2L, mu, nu)
        - (Expression.num(1) / Expression.num(4)) * FieldStrength(U1Y, mu, nu) * FieldStrength(U1Y, mu, nu)
    ),
)
show_model(sm_full_gauge_model)
for signature in sm_full_gauge_model.lagrangian().vertex_signatures():
    print(signature.names)


Lagrangian
1𝑖 * qL.bar * Gamma(mu) * CovD(qL, mu)

Feynman Rules
28 vertex signature(s)

Vertex: ('qL.bar', 'qL', 'G')
Rule: 1𝑖*gS*g(cof(2, w1),cof(2, w2))*gamma(bis(4, i1),bis(4, i2),mink(4, mu3))*t(coad(8, a3),cof(3, c1),cof(3, c2))

Vertex: ('qL.bar', 'qL', 'W')
Rule: 1𝑖*g2*g(cof(3, c1),cof(3, c2))*gamma(bis(4, i1),bis(4, i2),mink(4, mu3))*t(coad(3, aw3),cof(2, w1),cof(2, w2))

Vertex: ('qL.bar', 'qL', 'B')
Rule: 1𝑖/6*g1*g(cof(2, w1),cof(2, w2))*g(cof(3, c1),cof(3, c2))*gamma(bis(4, i1),bis(4, i2),mink(4, mu3))

Vertex: ('qL.bar', 'qL')
Rule: 1𝑖*g(cof(2, w1),cof(2, w2))*g(cof(3, c1),cof(3, c2))*gamma(bis(4, i1),bis(4, i2),mink(4, mu1_int))*pcomp(q2,mu1_int)

Vertex: ('uR.bar', 'uR', 'G')
Rule: 1𝑖*gS*gamma(bis(4, i1),bis(4, i2),mink(4, mu3))*t(coad(8, a3),cof(3, c1),cof(3, c2))

Vertex: ('uR.bar', 'uR', 'B')
Rule: 2𝑖/3*g1*g(cof(3, c1),cof(3, c2))*gamma(bis(4, i1),bis(4, i2),mink(4, mu3))

Vertex: ('uR.bar', 'uR')
Rule: 1𝑖*g(cof(3, c1),cof(3, c2))*gamma(bis(4, i1),bis(4, i2),mink(4, m

In [14]:
for signature in sm_gauge_model.lagrangian().vertex_signatures():
    print(signature.names)


('H.bar', 'H')
('dR.bar', 'dR')
('eR.bar', 'eR')
('lL.bar', 'lL')
('qL.bar', 'qL')
('uR.bar', 'uR')
('H.bar', 'H', 'B')
('H.bar', 'H', 'W')
('dR.bar', 'dR', 'B')
('dR.bar', 'dR', 'G')
('eR.bar', 'eR', 'B')
('lL.bar', 'lL', 'B')
('lL.bar', 'lL', 'W')
('qL.bar', 'qL', 'B')
('qL.bar', 'qL', 'G')
('qL.bar', 'qL', 'W')
('uR.bar', 'uR', 'B')
('uR.bar', 'uR', 'G')
('H.bar', 'H', 'B', 'B')
('H.bar', 'H', 'W', 'B')
('H.bar', 'H', 'W', 'W')


## Unbroken Higgs Potential


In [ ]:
muH2 = S("muH2")
lamH = S("lamH")

sm_higgs_potential_model = Model(
    gauge_groups=(SU2L, U1Y),
    fields=(H,),
    lagrangian_decl=muH2 * H.bar * H - lamH * (H.bar * H) * (H.bar * H),
)

for signature in sm_higgs_potential_model.lagrangian().vertex_signatures():
    print(signature.names)


## SU(2) WWWW Cross-Check


In [16]:
from symbolic.vertex_postprocessing import simplify_su2_ff

wwww_raw = su2_ym_model.lagrangian().feynman_rule(W, W, W, W, include_delta=False)
wwww_delta = simplify_su2_ff(wwww_raw)

eta12 = LORENTZ_INDEX.representation.g(S("mu1"), S("mu2")).to_expression()
eta13 = LORENTZ_INDEX.representation.g(S("mu1"), S("mu3")).to_expression()
eta14 = LORENTZ_INDEX.representation.g(S("mu1"), S("mu4")).to_expression()
eta23 = LORENTZ_INDEX.representation.g(S("mu2"), S("mu3")).to_expression()
eta24 = LORENTZ_INDEX.representation.g(S("mu2"), S("mu4")).to_expression()
eta34 = LORENTZ_INDEX.representation.g(S("mu3"), S("mu4")).to_expression()

delta14 = WEAK_ADJ_INDEX.representation.g(S("aw1"), S("aw4")).to_expression()
delta23 = WEAK_ADJ_INDEX.representation.g(S("aw2"), S("aw3")).to_expression()
delta13 = WEAK_ADJ_INDEX.representation.g(S("aw1"), S("aw3")).to_expression()
delta24 = WEAK_ADJ_INDEX.representation.g(S("aw2"), S("aw4")).to_expression()
delta12 = WEAK_ADJ_INDEX.representation.g(S("aw1"), S("aw2")).to_expression()
delta34 = WEAK_ADJ_INDEX.representation.g(S("aw3"), S("aw4")).to_expression()

wwww_feynrules = I * g2**2 * (
    eta14 * eta23 * (
        -2 * delta14 * delta23
        + delta13 * delta24
        + delta12 * delta34
    )
    + eta13 * eta24 * (
        delta14 * delta23
        - 2 * delta13 * delta24
        + delta12 * delta34
    )
    + eta12 * eta34 * (
        delta14 * delta23
        + delta13 * delta24
        - 2 * delta12 * delta34
    )
)

show("WWWW f*f basis", wwww_raw)
show("WWWW delta basis", wwww_delta)
show("WWWW FeynRules basis", wwww_feynrules)
show("WWWW difference", (wwww_delta - wwww_feynrules).expand())


PASS: checked 81 SU(2) components.
PASS: WWWW vertex matches the FeynRules SU(2) result.
